In [1]:
# Create an API client and Helper Functions
from anthropic import Anthropic
from dotenv import load_dotenv
import os

load_dotenv()

token = os.environ["ANTHROPIC_AUTH_TOKEN"]
url = os.environ["ANTHROPIC_BASE_URL"]
model = os.environ["ANTHROPIC_MODEL"]

client = Anthropic(
    api_key=token,
    base_url=url,
    default_headers={"Authorization": f"Bearer {token}"}
)

def add_user_message(messages, content):
    user_message = { "role": "user", "content": content }
    messages.append(user_message)

def add_assistant_message(messages, content):
    assistant_message = { "role": "assistant", "content": content }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0):

  params = {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature,
  }

  if system:
    params["system"] = system
  
  message = client.messages.create(**params)
  return message.content[0].text

In [8]:
# For this lesson, we'll ignore the chat function, as it does not work with streaming the way we implemented.

messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database.")

stream = client.messages.create(
  model=model,
  max_tokens=1000,
  messages=messages,
  stream=True
)

for event in stream:
  print(event, end="")
print("\n--------------------------")

# There is a lot of Common Events that can be returned in the stream.
# RawContentBlockDeltaEvent: This event contains the text content of the model's response. 
# It is sent in chunks, so you will receive multiple events for a single response.

messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database.")

with client.messages.stream(
  model=model,
  max_tokens=1000,
  messages=messages
) as stream:
  for text in stream.text_stream:
    print(text, end="")
print("\n--------------------------")

RawMessageStartEvent(message=Message(id='msg_011Ce3V9yY7KAuxi1kmjtGHZ', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=19, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')RawContentBlockDeltaEvent(delta=TextDelta(text='"', type='text_delta'), index=0, type='content_block_delta')RawContentBlockDeltaEvent(delta=TextDelta(text='MemoDB is a fictional cloud-based database system that stores imaginary user records, invented transactions', type='text_delta'), index=0, type='content_block_delta')RawContentBloc

In [9]:
# Often we want to stream a response so that the user can see each chunk of the stream as soon as possible.

with client.messages.stream(
  model=model,
  max_tokens=1000,
  messages=messages
) as stream:
  for text in stream.text_stream:
    pass

stream.get_final_message()

ParsedMessage(id='msg_011Ce3VHtE6nDrcmYVo3BNnJ', container=None, content=[ParsedTextBlock(citations=None, text='Here is a one sentence description of a fake database:\n\n**NebulaBase** is a fictional cloud-based database management system that stores and organizes intergalactic census data for over 47 billion registered alien species across 12 known galaxies.', type='text', parsed_output=None)], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=19, output_tokens=59, output_tokens_details=None, server_tool_use=None, service_tier='standard'))